In [12]:
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools, Descriptors, rdchem, rdMolDescriptors
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def mol_to_feat(mol):
    #找我认为的重要键
    NO_single = NO_double = 0
    NN_single = NN_double = 0
    sum = 0
    for i in mol.GetBonds():
        pre = i.GetBeginAtom().GetSymbol()
        nxt = i.GetEndAtom().GetSymbol()
        link = i.GetBondType()
        if pre == "O":
            pre, nxt = nxt, pre
        if pre == "N" and nxt == "O":
            if link == rdchem.BondType.SINGLE:
                NO_single += 1
            else:
                NO_double += 1
        elif pre == "N" and nxt == "N":
            if link == rdchem.BondType.SINGLE:
                NN_single += 1
            else:
                NN_double += 1
        sum += 1

    #算各种原子的个数
    mqn = rdMolDescriptors.MQNs_(mol)
    N = mqn[7] + mqn[8]
    O = mqn[9] + mqn[10]
    C = mqn[0]
    mol = Chem.AddHs(mol)
    H = mol.GetNumAtoms() - mol.GetNumHeavyAtoms()
    
    feat = [
        (C * 2 + H / 2 - O) * 16 / Descriptors.MolWt(mol),
        N * 14 / Descriptors.MolWt(mol),
        NO_single / sum, NO_double / sum,
        NN_single / sum, NN_double / sum
    ]
    return feat

file = Path.cwd().parent / "data" / "test.xlsx"
df = pd.read_excel(file)
PandasTools.AddMoleculeColumnToFrame(df, "SMILES", "ROMol", False)
x_all = np.stack(df["ROMol"].apply(mol_to_feat).tolist(), axis = 0)
y_all = np.array(df["Q(cal/g)"], dtype = np.float64)

for i in range(3):
    if i == 0:
        print("PolynomialRegression:")
        pipe = make_pipeline(PolynomialFeatures(4), StandardScaler(), LassoCV(cv = 5))
        scores = cross_validate(pipe, x_all, y_all, cv = 10, scoring = ["neg_mean_squared_error", "neg_mean_absolute_error", "r2"], n_jobs = -1)
    if i == 1:
        print("RandomForest:")
        rf = RandomForestRegressor(
            n_estimators = 200,
            n_jobs = -1,
            random_state = 323922
        )
        scores = cross_validate(rf, x_all, y_all, cv = 10, scoring = ["neg_mean_squared_error", "neg_mean_absolute_error", "r2"], n_jobs = -1)
    if i == 2:
        print("XGBoost:")
        clf = XGBRegressor(
            n_estimators = 1000,
            learning_rate = 0.05,
            reg_lambda = 1.0,
            n_jobs = -1,
            random_state = 323922
        )
        scores = cross_validate(clf, x_all, y_all, cv = 10, scoring = ["neg_mean_squared_error", "neg_mean_absolute_error", "r2"], n_jobs = -1)
    rmse = ((-scores["test_neg_mean_squared_error"]) ** 0.5).mean()
    mae = np.abs(scores["test_neg_mean_absolute_error"]).mean()
    r2 = scores["test_r2"].mean()
    print("rmse:", round(rmse, 2))
    print("mae:", round(mae, 2))
    print("r2:", round(r2, 2))
    print()

PolynomialRegression:
rmse: 169.86
mae: 133.65
r2: 0.8

RandomForest:
rmse: 182.3
mae: 152.13
r2: 0.75

XGBoost:
rmse: 185.33
mae: 148.98
r2: 0.75

